# LLM-based architecture search for time-series forecasting

Этот ноутбук выполняет полный экспериментальный цикл для задачи прогноза временных рядов (семейство датасетов ETT):

1. Загрузка и разбиение датасетов ETTm1, ETTh1, ETTh2.
2. Baseline с оптимизацией гиперпараметров LSTM-модели через Optuna.
3. Запуск оригинального Informer (репозиторий `Informer2020`) на тех же датасетах через внешний скрипт-обёртку.
4. LLM-агент, автоматически генерирующий архитектуры и скрипты обучения, выполняемые в изолированном раннере.
5. Автоматическое сравнение качества (MSE) и визуализация результатов.

Ноутбук опирается на код из пакета `edlm_search` и оригинальной реализации Informer, расположенной в каталоге `src/Informer2020`.


In [ ]:
from __future__ import annotations

import asyncio
import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
src_dir = project_root / 'src'

if not src_dir.exists():
    raise RuntimeError(f'Expected src directory at {src_dir}, but it does not exist.')

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from experiments import (
    run_lstm_ettm1_experiment,
    run_informer_ettm1_experiment,
    ExperimentResult,
)
from experiments.datasets import load_ett_csv_dataset
from edlm_search.baseline_optuna import run_optuna_for_ettm1
from edlm_search.candidates_database import CandidateDatabase
from edlm_search.ett_evaluator import ETTM1Evaluator
from edlm_search.llm_clients import DeepSeekClient, LMStudioClient
from edlm_search.problem import Problem
from edlm_search.runner import UnsafeRunner
from edlm_search.sampler import CandidateSampler
from edlm_search.search_loop import LLMBasedArchitectureSearch

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)
logger = logging.getLogger('main_notebook')

logger.info(f'Project root: {project_root}')
logger.info(f'src dir: {src_dir}')


In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DatasetPaths:
    '''Absolute paths to CSV files of the ETT family datasets.'''
    ettm1_csv_path: Path
    etth1_csv_path: Path
    etth2_csv_path: Path


@dataclass
class SplitConfig:
    '''Train/validation split configuration for ETT datasets.'''
    max_rows: int
    train_ratio: float


@dataclass
class OptunaConfig:
    '''Hyperparameters for the LSTM Optuna baseline search.'''
    seq_len: int
    pred_len: int
    num_epochs: int
    n_trials: int


@dataclass
class LLMSearchConfig:
    '''Configuration of the LLM-based architecture search loop.'''
    num_epochs: int
    max_candidates: int
    top_k_for_crossover: int
    metric_name: str


@dataclass
class InformerConfig:
    '''Configuration for running Informer via the external wrapper script.'''
    num_epochs: int


@dataclass
class LLMProviderConfig:
    '''Configuration of the LLM provider (LM Studio or DeepSeek).'''
    provider: str
    base_url: str
    model_name: str
    api_key_env_var: str


dataset_paths = DatasetPaths(
    ettm1_csv_path=project_root / 'ETDataset' / 'ETTm1.csv',
    etth1_csv_path=project_root / 'ETDataset' / 'ETTh1.csv',
    etth2_csv_path=project_root / 'ETDataset' / 'ETTh2.csv',
)

split_config = SplitConfig(
    max_rows=10000,
    train_ratio=0.8,
)

optuna_config = OptunaConfig(
    seq_len=96,
    pred_len=24,
    num_epochs=5,
    n_trials=20,
)

llm_search_config = LLMSearchConfig(
    num_epochs=5,
    max_candidates=5,
    top_k_for_crossover=3,
    metric_name='mse',
)

informer_config = InformerConfig(
    num_epochs=10,
)

provider_config = LLMProviderConfig(
    provider=os.environ.get('LLM_PROVIDER', 'lmstudio'),
    base_url=os.environ.get('LLM_BASE_URL', 'http://localhost:1234/v1'),
    model_name=os.environ.get('LLM_MODEL_NAME', 'your-lmstudio-model-name'),
    api_key_env_var='DEEPSEEK_API_KEY',
)

logger.info(f'LLM provider: {provider_config.provider}')
logger.info(f'ETTm1 path: {dataset_paths.ettm1_csv_path}')
logger.info(f'ETTh1 path: {dataset_paths.etth1_csv_path}')
logger.info(f'ETTh2 path: {dataset_paths.etth2_csv_path}')


In [ ]:
def load_datasets(dataset_paths: DatasetPaths, split_config: SplitConfig):
    '''Load and split ETTm1, ETTh1 and ETTh2 datasets into train and validation parts.'''
    train_dfs: dict[str, pd.DataFrame] = {}
    valid_dfs: dict[str, pd.DataFrame] = {}

    for name, csv_path in (
        ('ETTm1', dataset_paths.ettm1_csv_path),
        ('ETTh1', dataset_paths.etth1_csv_path),
        ('ETTh2', dataset_paths.etth2_csv_path),
    ):
        if not csv_path.exists():
            raise FileNotFoundError(f'Dataset file {name} not found at {csv_path}')
        train_df, valid_df = load_ett_csv_dataset(
            csv_path=str(csv_path),
            max_rows=split_config.max_rows,
            train_ratio=split_config.train_ratio,
        )
        logger.info(f'Dataset {name} loaded: train={len(train_df)}, valid={len(valid_df)}')
        train_dfs[name] = train_df
        valid_dfs[name] = valid_df

    return train_dfs, valid_dfs


train_dfs, valid_dfs = load_datasets(dataset_paths=dataset_paths, split_config=split_config)


In [ ]:
ettm1_train = train_dfs['ETTm1']
ettm1_valid = valid_dfs['ETTm1']

logger.info(
    f'Starting Optuna baseline on ETTm1: seq_len={optuna_config.seq_len}, '
    f'pred_len={optuna_config.pred_len}, epochs={optuna_config.num_epochs}, '
    f'trials={optuna_config.n_trials}'
)

optuna_study = run_optuna_for_ettm1(
    train_df=ettm1_train,
    valid_df=ettm1_valid,
    seq_len=optuna_config.seq_len,
    pred_len=optuna_config.pred_len,
    num_epochs=optuna_config.num_epochs,
    n_trials=optuna_config.n_trials,
)

optuna_best_mse = float(optuna_study.best_value)

logger.info(f'Optuna baseline finished. Best MSE: {optuna_best_mse}')
logger.info(f'Best hyperparameters: {optuna_study.best_trial.params}')


In [ ]:
informer_results: dict[str, ExperimentResult] = {}

for dataset_name, csv_path in (
    ('ETTm1', dataset_paths.ettm1_csv_path),
    ('ETTh1', dataset_paths.etth1_csv_path),
    ('ETTh2', dataset_paths.etth2_csv_path),
):
    if not csv_path.exists():
        logger.info(f'Skipping Informer run for {dataset_name}: file {csv_path} not found.')
        continue

    metrics_json_path = project_root / 'artifacts' / f'informer_{dataset_name.lower()}_metrics.json'
    informer_script = src_dir / 'Informer2020' / 'informer_experiment_wrapper.py'

    logger.info(
        f'Running Informer for {dataset_name}: script={informer_script}, csv={csv_path}, '
        f'epochs={informer_config.num_epochs}'
    )

    result = run_informer_ettm1_experiment(
        csv_path=str(csv_path),
        informer_script_path=str(informer_script),
        metrics_json_path=str(metrics_json_path),
        extra_args=['--epochs', str(informer_config.num_epochs), '--data', dataset_name],
        model_name=f'informer-{dataset_name.lower()}',
    )
    informer_results[dataset_name] = result
    logger.info(f'Informer for {dataset_name} finished. Metrics: {result.metrics}')


In [ ]:
def create_llm_pipeline(provider_config: LLMProviderConfig):
    '''Create an LLMPipeline instance for the configured provider.'''
    provider_lower = provider_config.provider.lower()

    if provider_lower == 'lmstudio':
        client = LMStudioClient(
            base_url=provider_config.base_url,
            model_name=provider_config.model_name,
        )
    elif provider_lower == 'deepseek':
        api_key = os.environ.get(provider_config.api_key_env_var)
        if not api_key:
            raise RuntimeError(
                f"Provider 'deepseek' requires environment variable {provider_config.api_key_env_var} to be set."
            )
        client = DeepSeekClient(
            api_key=api_key,
            base_url=provider_config.base_url,
            model_name=provider_config.model_name,
        )
    else:
        raise ValueError(f'Unknown LLM provider: {provider_config.provider}')

    return client.create_pipeline()


problem_dir = project_root / 'examples' / 'et'
if not problem_dir.exists():
    raise FileNotFoundError(f'Problem directory not found: {problem_dir}')

problem = Problem.from_directory(str(problem_dir))
logger.info('Problem statement successfully loaded.')

llm_pipeline = create_llm_pipeline(provider_config=provider_config)
sampler = CandidateSampler(llm_pipeline=llm_pipeline, problem=problem)

ettm1_train_df = train_dfs['ETTm1']
ettm1_valid_df = valid_dfs['ETTm1']

evaluator = ETTM1Evaluator(
    train_df=ettm1_train_df,
    valid_df=ettm1_valid_df,
    target_column='OT',
    metric_name=llm_search_config.metric_name,
    num_epochs=llm_search_config.num_epochs,
)

def runner_factory():
    '''Return a new UnsafeRunner instance for candidate execution.'''
    return UnsafeRunner()


search = LLMBasedArchitectureSearch(
    evaluator=evaluator,
    sampler=sampler,
    runner_factory=runner_factory,
    backend_name=provider_config.provider,
    metric_name=llm_search_config.metric_name,
    max_candidates=llm_search_config.max_candidates,
    top_k_for_crossover=llm_search_config.top_k_for_crossover,
)

logger.info(
    f'Starting LLM-based architecture search: max_candidates={llm_search_config.max_candidates}, '
    f'epochs_per_candidate={llm_search_config.num_epochs}'
)

search_database = asyncio.run(search.run_search())

logger.info(f'LLM-based search finished. Total candidates in database: {len(search_database)}')


In [ ]:
llm_best_records = search_database.top_k_by_metric(
    metric_name=llm_search_config.metric_name,
    k=1,
)
llm_best_record = llm_best_records[0]
llm_best_mse = float(llm_best_record.metrics[llm_search_config.metric_name])

logger.info(
    f'Best LLM candidate: id={llm_best_record.candidate_id}, '
    f'{llm_search_config.metric_name}={llm_best_mse}'
)

llm_search_df = search_database.to_dataframe()
llm_search_df


In [ ]:
rows: list[dict[str, object]] = []

rows.append(
    {
        'approach': 'lstm_optuna_best',
        'dataset': 'ETTm1',
        'mse': optuna_best_mse,
    }
)

for name, result in informer_results.items():
    mse_value = result.metrics.get('mse')
    rows.append(
        {
            'approach': 'informer_original',
            'dataset': name,
            'mse': float(mse_value) if mse_value is not None else np.nan,
        }
    )

rows.append(
    {
        'approach': 'llm_search_best_candidate',
        'dataset': 'ETTm1',
        'mse': llm_best_mse,
    }
)

comparison_df = pd.DataFrame(rows)
comparison_df.sort_values(by=['dataset', 'approach'], inplace=True)
comparison_df.reset_index(drop=True, inplace=True)

logger.info('Final comparison table between approaches is ready.')
comparison_df


In [ ]:
plt.figure(figsize=(8, 4))
x_positions = np.arange(len(comparison_df))
plt.bar(x_positions, comparison_df['mse'])
plt.xticks(
    x_positions,
    [f'{row.dataset}\n{row.approach}' for row in comparison_df.itertuples()],
    rotation=45,
    ha='right',
)
plt.ylabel('MSE')
plt.title('Model comparison: MSE across approaches')
plt.tight_layout()
plt.show()


In [ ]:
if 'mse' in llm_search_df.columns:
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(llm_search_df) + 1), llm_search_df['mse'])
    plt.xlabel('Candidate index')
    plt.ylabel('MSE')
    plt.title('LLM search dynamics: MSE per candidate')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    logger.info("Column 'mse' is missing in the LLM search DataFrame; skipping dynamics plot.")
